# Лабораторная №4. Файлы и стандартная библиотека

**Фокус Python:** Path, with, UTF-8, CSV, JSON, Counter, defaultdict, deque, ZIP, argparse.

**Подготовка:** предыдущие работы; их готовый код переносить не требуется. Python 3.11+. Обязательная практика использует стандартную библиотеку.
**Объём:** шесть небольших заданий; ориентир 90–150 минут с изучением примеров. Время ориентировочное, корректируется после первого проведения.

## Порядок работы

1. Прочитайте объяснение и **предскажите вывод примера до запуска**.
2. Запустите пример, сравните с прогнозом.
3. Реализуйте задание в своей ячейке; заготовки намеренно не завершены.
4. Запустите самопроверку и попробуйте иной вход. Не подставляйте ожидаемые ответы вместо вычислений.
5. Ответьте себе на короткий вопрос после задания. На защите будет похожая проверка и небольшое изменение кода.

Открытые проверки показывают маленькие примеры контракта и помогают учиться. Они не проверяют все случаи и не заменяют вашу реализацию. До сдачи добавьте **два собственных граничных случая**. Для работ 1–2 временно замените входы в рабочей ячейке, запустите только её и проверьте результат вручную; фиксированные самопроверки относятся к исходным данным, после эксперимента восстановите их.

Документация и помощь при подготовке допустимы. Любую заимствованную конструкцию нужно суметь объяснить и изменить самостоятельно. Слишком сложный непонятный код замените понятным. На защите проверяется понимание, а не память о точных числах.

**Оценка:** практика 12 баллов (6 × 2), объяснение 2, прогноз результата 2, небольшое изменение 4. Подробную рубрику объявляет преподаватель до сдачи.


In [1]:
from pathlib import Path
import tempfile, csv, json, zipfile, argparse
from collections import Counter, defaultdict, deque

## 1. Текстовый файл

Path описывает путь; оператор / соединяет части. mkdir создаёт каталог. with закрывает файл и при ошибке. encoding="utf-8" задаёт кодировку. Относительные пути зависят от рабочего каталога ядра, поэтому тесты используют отдельную временную папку.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [2]:
with tempfile.TemporaryDirectory() as _demo_dir:
    _demo_path = Path(_demo_dir) / "hello.txt"
    _demo_path.write_text("Привет", encoding="utf-8")
    print(_demo_path.read_text(encoding="utf-8"))

Привет


### Самостоятельно — 2 балла

write_lines(lines, path) создаёт родителей и записывает каждую строку с завершающим \n, возвращает path. Вход — список строк без переводов строк. Пустой список создаёт пустой файл. read_lines(path) читает UTF-8 через with и возвращает splitlines().

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [3]:
def write_lines(lines: list[str], path: Path) -> Path:
    file_path = Path(path)
    file_path.parent.mkdir(parents = True, exist_ok = True)
    with file_path.open('w', encoding = 'utf-8') as f:
        for line in lines:
            f.write(f'{line}\n')
    return file_path
    raise NotImplementedError

def read_lines(path: Path) -> list[str]:
    file_path = Path(path)
    with file_path.open('r', encoding = 'utf-8') as f:
        return [line.rstrip('\n') for line in f]
    raise NotImplementedError

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [4]:
with tempfile.TemporaryDirectory() as folder:
    p = Path(folder) / "nested" / "notes.txt"
    assert write_lines(["Привет", ""], p) == p
    assert read_lines(p) == ["Привет", ""]
    write_lines([], p)
    assert p.read_bytes() == b""

**Проверьте понимание:** Чем режим w отличается от a?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 2. CSV с двумя полями

csv.DictReader связывает заголовки с текстовыми значениями. csv учитывает кавычки и запятые внутри полей; split(",") этого не умеет. newline="" оставляет обработку переводов строк модулю csv. Числа нужно преобразовывать явно.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [5]:
import io
print(list(csv.DictReader(io.StringIO("name,count\npen,2\n"))))

[{'name': 'pen', 'count': '2'}]


### Самостоятельно — 2 балла

load_items(path) читает CSV с заголовками name,count и возвращает список словарей с теми же ключами. name очистить strip; count преобразовать в int. Пустое имя, отрицательное число или неверный заголовок — ValueError. Порядок заголовков свободный, дополнительных колонок нет. Строки имеют два поля; файл только с заголовком даёт [].

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [6]:
def load_items(path: Path) -> list[dict]:
    ans = list(csv.DictReader(read_lines(path)))
    for dicts in ans:
        try:    
            cnt = int(dicts['count'])
        except ValueError:
            raise ValueError('count должен быть числом')
        if cnt < 0:
            raise ValueError('count не должен быть отрицательным')
        nm = dicts['name'].strip()
        if not nm:
            raise ValueError('имя не должно быть пустым')
        dicts['count'] = cnt
        dicts['name'] = nm
    return ans
    raise NotImplementedError


**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [7]:
with tempfile.TemporaryDirectory() as folder:
    p = Path(folder) / "items.csv"
    p.write_text('name,count\n"pen, blue",2\nbook,0\n', encoding="utf-8")
    assert load_items(p) == [{"name": "pen, blue", "count": 2}, {"name": "book", "count": 0}]
    p.write_text("name,count\n", encoding="utf-8")
    assert load_items(p) == []

**Проверьте понимание:** Почему count сначала строка?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 3. Удобные накопители

Counter считает частоты; defaultdict(int) даёт ноль для нового ключа; deque(maxlen=2) хранит два последних элемента. Это разные результаты: число записей и сумма count не взаимозаменяемы.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [8]:
print(Counter("aba"))
_demo_tail = deque(maxlen=2)
_demo_tail.extend([1, 2, 3])
print(list(_demo_tail))

Counter({'a': 2, 'b': 1})
[2, 3]


### Самостоятельно — 2 балла

summarize_items(items) возвращает {"rows_by_name": частоты имён, "count_by_name": суммы count, "last_names": последние два имени во входном порядке}. Используйте три названных контейнера, верните обычные dict и list. Корректный вход, пустой список даёт {}, {}, [].

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [9]:
def summarize_items(items: list[dict]) -> dict:
    result = dict()
    names = [dict['name'] for dict in items]
    cnt = Counter(names)
    result.setdefault('rows_by_name', dict(cnt))

    sums = {}
    for dicts in items:
        name = dicts['name']
        count = dicts['count']
        if name not in sums:
            sums[name] = 0
        sums[name] += count
    result.setdefault('count_by_name', sums)

    tail = deque(maxlen = 2)
    for dicts in items:
        tail.append(dicts['name'])
    result.setdefault('last_names', list(tail))
    return result
    raise NotImplementedError

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [10]:
sample = [{"name": "a", "count": 2}, {"name": "a", "count": 3}, {"name": "b", "count": 0}]
result = summarize_items(sample)
print(result)
assert result["rows_by_name"] == {"a": 2, "b": 1}
assert result["count_by_name"] == {"a": 5, "b": 0}
assert result["last_names"] == ["a", "b"]
assert summarize_items([]) == {"rows_by_name": {}, "count_by_name": {}, "last_names": []}

{'rows_by_name': {'a': 2, 'b': 1}, 'count_by_name': {'a': 5, 'b': 0}, 'last_names': ['a', 'b']}


**Проверьте понимание:** Почему для a частота 2, а количество 5?
Потому что в качестве ключа он встречается дважды, а сумма по count равна 5

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 4. JSON и обратное чтение

JSON хранит словари со строковыми ключами, списки и простые значения; Path и datetime напрямую не поддерживаются. json.dumps возвращает строку, json.loads восстанавливает данные. ensure_ascii=False оставляет кириллицу читаемой. Проверяйте восстановленную структуру, не только наличие файла.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [22]:
_demo_json = json.dumps({"имя": "Аня"}, ensure_ascii=False)
print(_demo_json, json.loads(_demo_json))

{"имя": "Аня"} {'имя': 'Аня'}


### Самостоятельно — 2 балла

save_report(report, path) сначала сериализует JSON-совместимый словарь с indent=2, ensure_ascii=False, allow_nan=False, затем создаёт родителей и записывает UTF-8; возвращает path. Повторная запись заменяет файл. Ошибки не подавлять.

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [12]:
def save_report(report: dict, path: Path) -> Path:
    djson = json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False)
    file_path = Path(path)
    file_path.parent.mkdir(parents = True, exist_ok = True)
    with file_path.open('w', encoding='utf-8') as f:
        f.write(djson)
    return file_path
    raise NotImplementedError

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [ ]:
with tempfile.TemporaryDirectory() as folder:
    p = Path(folder) / "out" / "report.json"
    data = {"имя": "Аня", "count": 2}
    assert save_report(data, p) == p
    assert json.loads(p.read_text(encoding="utf-8")) == data
    assert "Аня" in p.read_text(encoding="utf-8")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\7272~1\\AppData\\Local\\Temp\\tmpl_p2cwuv\\out\\report.json'

**Проверьте понимание:** Почему сериализуем до открытия файла на запись?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 5. Список файлов и ZIP

glob("*.txt") ищет совпадения в одном каталоге. is_file() отделяет файлы от каталогов. sorted() задаёт воспроизводимый порядок. arcname определяет имя внутри ZIP; namelist() и read() позволяют проверить архив.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [14]:
print(Path("folder/note.txt").name, Path("folder/note.txt").suffix)

note.txt .txt


### Самостоятельно — 2 балла

archive_texts(root, target) упаковывает только обычные .txt-файлы непосредственно в root, без рекурсии и без символических ссылок. Внутри ZIP — только имена файлов по алфавиту. root существует; target имеет суффикс .zip. Создайте родителей target, режим w, ZIP_DEFLATED. Верните список включённых имён.

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [15]:
def archive_texts(root: Path, target: Path) -> list[str]:
    root_path = Path(root)
    target_path = Path(target)
    target_path.parent.mkdir(parents=True, exist_ok=True)

    files_to_zip = []
    for elem in root_path.iterdir():
        if elem.is_file() and not elem.is_symlink() and elem.suffix == '.txt':
            files_to_zip.append(elem)
    files_to_zip.sort(key=lambda p: p.name)

    included_names = []
    with zipfile.ZipFile(target_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in files_to_zip:
            zf.write(file_path, arcname=file_path.name)
            included_names.append(file_path.name)
    print(included_names)
    return included_names
    raise NotImplementedError

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [16]:
with tempfile.TemporaryDirectory() as folder:
    root = Path(folder)
    (root / "b.txt").write_text("Б", encoding="utf-8")
    (root / "a.txt").write_text("А", encoding="utf-8")
    (root / "skip.csv").write_text("x", encoding="utf-8")
    target = root / "out" / "bundle.zip"
    assert archive_texts(root, target) == ["a.txt", "b.txt"]
    with zipfile.ZipFile(target) as archive:
        assert archive.namelist() == ["a.txt", "b.txt"]
        assert archive.read("a.txt") == (root / "a.txt").read_bytes()

['a.txt', 'b.txt']


**Проверьте понимание:** Почему путь на диске и arcname могут отличаться?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 6. Параметры запуска и маленький конвейер

argparse превращает список аргументов в Namespace. В Jupyter передавайте явный список в parse_args([...]), иначе получите служебные аргументы ядра. Конвейер соединяет функции, не переписывая каждую заново.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [17]:
_demo_parser = argparse.ArgumentParser()
_demo_parser.add_argument("--name", default="guest")
print(_demo_parser.parse_args(["--name", "Аня"]).name)

Аня


### Самостоятельно — 2 балла

build_parser(): обязательные --input и --output типа Path. run_report(args): разобрать явный список, вызвать load_items, summarize_items, save_report и вернуть сводку. Вызывайте предыдущие функции.

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [18]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description='Generate report from CSV')
    parser.add_argument('--input', type = Path, required=True, help = 'Path to input CSV file')
    parser.add_argument('--output', type = Path, required=True, help = 'Path to output CSV file')
    return parser
    raise NotImplementedError

def run_report(args: list[str]) -> dict:
    parser = build_parser()
    parsed_args = parser.parse_args(args)
    items = load_items(parsed_args.input)
    summary = summarize_items(items)
    save_report(summary, parsed_args.output)
    return summary
    raise NotImplementedError

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [19]:
with tempfile.TemporaryDirectory() as folder:
    source, target = Path(folder) / "in.csv", Path(folder) / "out.json"
    source.write_text("name,count\na,2\na,1\n", encoding="utf-8")
    report = run_report(["--input", str(source), "--output", str(target)])
    assert report == json.loads(target.read_text(encoding="utf-8"))
    assert report["count_by_name"] == {"a": 3}

**Проверьте понимание:** Зачем передавать args в run_report, а не читать глобальные настройки?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## Перед сдачей

- Реализованы шесть заданий; выполняются с чистого ядра.
- Добавлены два собственных граничных случая и объяснено, что они проверяют.
- Вы можете назвать тип результата, объяснить одну важную строку и изменить условие.
- Необязательное усложнение: придумайте одно расширение контракта и тест к нему. Оно не требуется для максимальной оценки.

### Три коротких наблюдения

1. Я ожидал(а) …, а получилось …, потому что …
2. Я исправил(а) ошибку … и проверил(а) это на …
3. Я пока путаю … / могу объяснить …

Заполните этот Markdown-блок своими словами. На защите преподаватель выберет один небольшой участок, а не попросит пересказывать всю работу.
